[Reference](https://python.plainenglish.io/track-request-performance-in-fastapi-with-one-simple-middleware-b45515545bda)

In [1]:
import time
from fastapi import FastAPI, Request

app = FastAPI()
@app.middleware("http")
async def add_process_time_header(request: Request, call_next):
    start_time = time.perf_counter()
    response = await call_next(request)
    process_time = time.perf_counter() - start_time
    response.headers["X-Process-Time"] = f"{process_time:.4f}"
    return response

```
curl -I http://localhost:8000/api/users
```

# Add Logging

In [3]:
from loguru import logger

@app.middleware("http")
async def add_process_time_header(request: Request, call_next):
    start_time = time.perf_counter()
    response = await call_next(request)
    process_time = time.perf_counter() - start_time

    response.headers["X-Process-Time"] = f"{process_time:.4f}"

    logger.info(
        f"{request.method} {request.url.path} - {process_time:.4f}s"
    )

    return response

# Catch Slow Requests

In [4]:
SLOW_THRESHOLD = 1.0  # seconds

@app.middleware("http")
async def add_process_time_header(request: Request, call_next):
    start_time = time.perf_counter()
    response = await call_next(request)
    process_time = time.perf_counter() - start_time

    response.headers["X-Process-Time"] = f"{process_time:.4f}"

    if process_time > SLOW_THRESHOLD:
        logger.warning(
            f"SLOW: {request.method} {request.url.path} - {process_time:.4f}s"
        )

    return response